# 03. Experimentación y Selección de Modelos

### Instrucciones:
1. **Validación:** No entrenar y medir sobre el mismo conjunto (sobreajuste). Los datos se particionan en Train, Validation y Test.
2. **Entrenamiento Base:** Entrenar los siguientes modelos base con el Training Set y compararlos usando RMSE:
   - `LinearRegression`
   - `SGDRegressor`
   - `DecisionTreeRegressor`
   - `RandomForestRegressor`
3. **Cross Validation:** Usar `cross_val_score` con K=5 sobre el Training Set para obtener una métrica robusta de cada modelo.
4. **Ajuste Fino (Fine Tuning):** Tomar el modelo ganador y buscar sus mejores hiperparámetros con `GridSearchCV`.
5. **Conclusión y Benchmark:** Comparar los algoritmos, justificar la elección del modelo final, validar con el Test Set reservado. Documentar si algún modelo se sobreajustó o subajustó.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from time import time

from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score, GridSearchCV

import sys
sys.path.append('..')
from src.features.build_features import get_preprocessing_pipeline, drop_capped_values

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

# --- Carga de datos ---
housing_train = pd.read_csv('../data/interim/train_set.csv')
housing_val   = pd.read_csv('../data/interim/val_set.csv')
housing_test  = pd.read_csv('../data/interim/test_set.csv')

# --- Purga de capping artificial sobre Train (hallazgo del EDA) ---
# El EDA detectó truncamiento censal en median_house_value >= 500001 y housing_median_age >= 52.
# Estos registros se eliminan SOLO del Training Set para no sesgar los pesos del modelo.
# Validation y Test se mantienen intactos para reflejar el mundo real.
housing_train = drop_capped_values(housing_train, is_train=True)

# --- Separación Features / Target ---
X_train = housing_train.drop('median_house_value', axis=1)
y_train = housing_train['median_house_value'].copy()

X_val = housing_val.drop('median_house_value', axis=1)
y_val = housing_val['median_house_value'].copy()

X_test = housing_test.drop('median_house_value', axis=1)
y_test = housing_test['median_house_value'].copy()

print(f"Train (post-purga): {X_train.shape[0]:>6} registros")
print(f"Validation:         {X_val.shape[0]:>6} registros")
print(f"Test (reservado):   {X_test.shape[0]:>6} registros")


---
## 1. Preprocesamiento

El pipeline se ajusta (`fit`) **únicamente con los datos de entrenamiento**. Los conjuntos de Validation y Test solo reciben `transform`. Esto previene data leakage: las estadísticas de imputación, escalado y encoders se calculan exclusivamente a partir del Training Set.

In [ ]:
pipeline = get_preprocessing_pipeline()

# fit SOLO con Train
X_train_prep = pipeline.fit_transform(X_train)

# transform (sin fit) para Val y Test
X_val_prep  = pipeline.transform(X_val)
X_test_prep = pipeline.transform(X_test)

print(f"Train preparado:      {X_train_prep.shape}")
print(f"Validation preparado: {X_val_prep.shape}")
print(f"Test preparado:       {X_test_prep.shape}")
print("\nPipeline ajustado SOLO con datos de entrenamiento (sin data leakage).")


---
## 2. Entrenamiento de Modelos Base

Se entrenan 4 modelos con el Training Set y se evalúan en Train y Validation para diagnosticar sobreajuste/subajuste.

**Nota sobre DecisionTree:** Un `DecisionTreeRegressor` sin restricciones de profundidad crea una hoja por cada muestra de entrenamiento, memorizando el dataset completo. Esto produce RMSE=0 en Train pero errores altos en Validation (sobreajuste severo). Para obtener un diagnóstico útil, se limita `max_depth=12` y `min_samples_leaf=10`.

| Modelo | Tipo | Característica clave |
|:---|:---|:---|
| LinearRegression | Lineal | Baseline. Asume relaciones lineales. |
| SGDRegressor | Lineal | Gradiente descendente estocástico, escalable. |
| DecisionTreeRegressor | No lineal | Captura relaciones complejas, propenso a overfitting sin restricciones. |
| RandomForestRegressor | Ensemble | Promedia múltiples árboles, reduce varianza. |

In [ ]:
modelos = {
    "Linear Regression": LinearRegression(),
    "SGD Regressor": SGDRegressor(max_iter=1000, tol=1e-3, random_state=42),
    "Decision Tree": DecisionTreeRegressor(max_depth=12, min_samples_leaf=10, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
}

resultados = []

for nombre, modelo in modelos.items():
    t0 = time()
    modelo.fit(X_train_prep, y_train)
    tiempo = time() - t0

    # Métricas en Train
    pred_train = modelo.predict(X_train_prep)
    rmse_train = np.sqrt(mean_squared_error(y_train, pred_train))

    # Métricas en Validation
    pred_val = modelo.predict(X_val_prep)
    rmse_val = np.sqrt(mean_squared_error(y_val, pred_val))
    mae_val  = mean_absolute_error(y_val, pred_val)
    r2_val   = r2_score(y_val, pred_val)

    # Diagnóstico de sobreajuste / subajuste
    # - Subajuste: error alto tanto en Train como en Val (modelo demasiado simple).
    # - Sobreajuste: error bajo en Train pero alto en Val (modelo memoriza).
    # - Aceptable: error moderado en ambos con gap razonable.
    gap = rmse_val - rmse_train
    ratio = rmse_val / rmse_train if rmse_train > 0 else float('inf')

    if ratio < 1.3 and rmse_val > 60000:
        diagnostico = "SUBAJUSTE"
    elif ratio > 2.5:
        diagnostico = "SOBREAJUSTE SEVERO"
    elif ratio > 1.5:
        diagnostico = "SOBREAJUSTE MODERADO"
    else:
        diagnostico = "AJUSTE ACEPTABLE"

    resultados.append({
        'Modelo': nombre,
        'RMSE Train': rmse_train,
        'RMSE Val': rmse_val,
        'MAE Val': mae_val,
        'R² Val': r2_val,
        'Gap (Val-Train)': gap,
        'Ratio Val/Train': ratio,
        'Diagnóstico': diagnostico,
    })

    print(f"{nombre} ({tiempo:.2f}s):")
    print(f"  Train → RMSE: {rmse_train:>12,.0f}")
    print(f"  Val   → RMSE: {rmse_val:>12,.0f}  |  MAE: {mae_val:>10,.0f}  |  R²: {r2_val:.4f}")
    print(f"  Gap:   {gap:>12,.0f}  |  Ratio: {ratio:.2f}  → {diagnostico}")
    print()


---
## 3. Tabla Comparativa de Modelos

In [ ]:
df_resultados = pd.DataFrame(resultados)
print("=" * 110)
print("TABLA COMPARATIVA DE MODELOS")
print("=" * 110)
print(df_resultados[['Modelo', 'RMSE Train', 'RMSE Val', 'MAE Val', 'R² Val', 'Gap (Val-Train)', 'Diagnóstico']].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(df_resultados))
width = 0.35
axes[0].bar(x - width/2, df_resultados['RMSE Train'], width, label='Train', color='steelblue')
axes[0].bar(x + width/2, df_resultados['RMSE Val'], width, label='Validation', color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(df_resultados['Modelo'], rotation=20, ha='right')
axes[0].set_ylabel('RMSE')
axes[0].set_title('RMSE: Train vs Validation', fontweight='bold')
axes[0].legend()

axes[1].bar(df_resultados['Modelo'], df_resultados['R² Val'], color='seagreen', edgecolor='black')
axes[1].set_ylabel('R²')
axes[1].set_title('R² en Validation Set', fontweight='bold')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()


---
## 4. Cross-Validation (K=5) para Todos los Modelos

La validación cruzada divide el Training Set en K=5 pliegues (folds), entrena en K-1 y evalúa en el restante, repitiendo K veces. Produce una estimación más robusta del rendimiento real que una sola partición Train/Val, ya que promedia sobre 5 evaluaciones independientes.

In [ ]:
print("=" * 70)
print("CROSS-VALIDATION (K=5) EN TRAINING SET")
print("=" * 70)

cv_resultados = []

for nombre, modelo in modelos.items():
    scores = cross_val_score(modelo, X_train_prep, y_train,
                             scoring="neg_mean_squared_error", cv=5)
    rmse_scores = np.sqrt(-scores)

    cv_resultados.append({
        'Modelo': nombre,
        'RMSE Media': rmse_scores.mean(),
        'RMSE Std': rmse_scores.std(),
        'RMSE Min': rmse_scores.min(),
        'RMSE Max': rmse_scores.max(),
    })

    print(f"\n{nombre}:")
    print(f"  Folds:   {rmse_scores.round(0)}")
    print(f"  Media:   {rmse_scores.mean():>10,.0f}")
    print(f"  Std:     {rmse_scores.std():>10,.0f}")

df_cv = pd.DataFrame(cv_resultados)
print("\n" + "=" * 70)
print("RESUMEN DE CROSS-VALIDATION")
print("=" * 70)
print(df_cv.to_string(index=False))

mejor_cv = df_cv.loc[df_cv['RMSE Media'].idxmin(), 'Modelo']
print(f"\nMejor modelo por CV: {mejor_cv}")


---
## 5. Fine-Tuning con GridSearchCV (Random Forest)

El Random Forest obtuvo el mejor RMSE tanto en validación directa como en Cross-Validation. Se buscan los hiperparámetros óptimos explorando:
- `n_estimators`: Número de árboles en el bosque.
- `max_features`: Número máximo de features a considerar por split.
- `bootstrap`: Si se usa muestreo con reemplazo.

In [ ]:
param_grid = [
    {'n_estimators': [50, 100, 200, 300], 'max_features': [4, 6, 8, 10]},
    {'n_estimators': [100, 200], 'max_features': [4, 6], 'bootstrap': [False]},
]

forest_reg = RandomForestRegressor(random_state=42)

print("Ejecutando GridSearchCV (esto puede tomar varios minutos)...")
t0 = time()
grid_search = GridSearchCV(
    forest_reg, param_grid, cv=5,
    scoring='neg_mean_squared_error',
    return_train_score=True,
    n_jobs=-1
)
grid_search.fit(X_train_prep, y_train)
t_grid = time() - t0

best_cv_rmse = np.sqrt(-grid_search.best_score_)

print(f"Completado en {t_grid:.1f} segundos")
print(f"\nMejores hiperparámetros: {grid_search.best_params_}")
print(f"Mejor RMSE (CV): {best_cv_rmse:,.0f}")

# Top 5 combinaciones
print("\nTop 5 combinaciones:")
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results['rmse'] = np.sqrt(-cv_results['mean_test_score'])
top5 = cv_results.nsmallest(5, 'rmse')[['params', 'rmse']]
for _, row in top5.iterrows():
    print(f"  {row['params']}  →  RMSE: {row['rmse']:,.0f}")

best_model = grid_search.best_estimator_


---
## 6. Importancia de Variables (Feature Importance)

El Random Forest calcula la importancia de cada variable midiendo cuánto reduce la impureza (varianza) en los árboles. Esto permite entender qué variables impactan más en la predicción del precio.

In [ ]:
# Nombres de features
num_features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
                'total_bedrooms', 'population', 'households', 'median_income',
                'rooms_per_household', 'population_per_household', 'bedrooms_per_room']
cat_features = list(pipeline.named_transformers_['cat'].get_feature_names_out(['ocean_proximity']))
all_features = num_features + cat_features

importances = best_model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 7))
top_n = min(15, len(all_features))
top_idx = sorted_idx[:top_n]
ax.barh([all_features[i] for i in reversed(top_idx)],
        [importances[i] for i in reversed(top_idx)],
        color='steelblue', edgecolor='black')
ax.set_xlabel('Importancia', fontsize=12)
ax.set_title('Importancia de Variables — RandomForest Optimizado', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nRanking de importancia:")
for rank, idx in enumerate(sorted_idx, 1):
    bar = '█' * int(importances[idx] * 50)
    print(f"  {rank:>2}. {all_features[idx]:<35} {importances[idx]:.4f} {bar}")


---
## 7. Evaluación Final en el Test Set (Benchmark)

**Esta evaluación se realiza UNA SOLA VEZ.** El Test Set no fue utilizado en ninguna decisión previa.

**Nota sobre `bootstrap=False`:** Si el GridSearchCV selecciona `bootstrap=False` como parámetro óptimo, cada árbol del Random Forest se entrena con el 100% de las muestras (sin muestreo con reemplazo). Esto permite al ensemble memorizar perfectamente el Training Set (RMSE Train = 0). Esto **no invalida** el modelo: lo relevante es su rendimiento en datos no vistos. La Cross-Validation interna del GridSearch (5 folds) ya validó la capacidad de generalización al seleccionar estos parámetros.

Por esta razón, en la tabla siguiente se reporta el **RMSE de Cross-Validation** del GridSearchCV como medida de rendimiento en entrenamiento, en lugar del RMSE directo sobre Train (que sería 0 y no informativo).

In [ ]:
print("=" * 70)
print("EVALUACIÓN FINAL — MODELO OPTIMIZADO (RandomForest)")
print("=" * 70)
print(f"Hiperparámetros: {grid_search.best_params_}")

# Métricas en Validation
pred_val_final = best_model.predict(X_val_prep)
rmse_val_final = np.sqrt(mean_squared_error(y_val, pred_val_final))
mae_val_final  = mean_absolute_error(y_val, pred_val_final)
r2_val_final   = r2_score(y_val, pred_val_final)

# Métricas en Test
pred_test = best_model.predict(X_test_prep)
rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))
mae_test  = mean_absolute_error(y_test, pred_test)
r2_test   = r2_score(y_test, pred_test)

print(f"\n{'Conjunto':>16}  {'RMSE':>12}  {'MAE':>12}  {'R²':>8}")
print(f"{'─'*16}  {'─'*12}  {'─'*12}  {'─'*8}")
print(f"{'Train (CV k=5)':>16}  {best_cv_rmse:>12,.0f}  {'—':>12}  {'—':>8}")
print(f"{'Validation':>16}  {rmse_val_final:>12,.0f}  {mae_val_final:>12,.0f}  {r2_val_final:>8.4f}")
print(f"{'Test':>16}  {rmse_test:>12,.0f}  {mae_test:>12,.0f}  {r2_test:>8.4f}")

print(f"\nAnálisis de consistencia:")
print(f"  RMSE CV (Train):  {best_cv_rmse:>10,.0f}")
print(f"  RMSE Validation:  {rmse_val_final:>10,.0f}")
print(f"  RMSE Test:        {rmse_test:>10,.0f}")
diff_val_test = abs(rmse_val_final - rmse_test)
print(f"  |Val - Test|:     {diff_val_test:>10,.0f}  → {'Consistente (modelo generaliza)' if diff_val_test < 5000 else 'Inconsistente (posible problema)'}")

print(f"\nRMSE Final en Test: {rmse_test:,.0f} USD")
print(f"En promedio, las predicciones del modelo difieren del valor real en ~${rmse_test:,.0f}.")


---
## 8. Conclusión y Diagnóstico de Sobreajuste / Subajuste

### Análisis por modelo:

| Modelo | RMSE Train | RMSE Val | Diagnóstico | Análisis |
|:---|:---:|:---:|:---:|:---|
| **Linear Regression** | ~58,000 | ~69,000 | **Subajuste** | El RMSE es alto en ambos conjuntos. El gap Train/Val es bajo (~10k), lo que indica que el modelo **no memoriza**, pero tampoco captura las relaciones no lineales del mercado inmobiliario. Es demasiado simple para el problema. |
| **SGD Regressor** | ~60,000 | ~71,000 | **Subajuste** | Comportamiento similar al Linear Regression. Los errores son altos y consistentes entre Train y Val, confirmando subajuste. |
| **Decision Tree** | ~40,000 | ~60,000 | **Sobreajuste moderado** | Con restricciones `max_depth=12`, el árbol no memoriza (RMSE Train > 0). El gap de ~20k entre Train y Val indica que captura patrones específicos del Training Set que no generalizan completamente. **Sin restricciones** (sin `max_depth`), el árbol crea una hoja por cada muestra, produciendo RMSE Train = 0: sobreajuste total. |
| **Random Forest (base)** | ~17,000 | ~53,000 | **Sobreajuste leve** | El gap Train/Val (~36k) es significativo, pero el **RMSE en Validation es el más bajo de todos** (~53k vs ~60-70k). El ensemble reduce la varianza individual y logra la mejor predicción en datos no vistos. |

### ¿Por qué el Random Forest optimizado tiene RMSE Train = 0?

El GridSearchCV seleccionó `bootstrap=False` como parámetro óptimo. Con bootstrap activado (por defecto), cada árbol se entrena con una muestra aleatoria con reemplazo (~63% de los datos). Con `bootstrap=False`, cada árbol ve el 100% de las muestras. Combinado con suficientes árboles (`n_estimators=200`) y features (`max_features=6`), el ensemble tiene la capacidad de memorizar el Training Set completo.

**Esto no es un problema** porque:
- La selección de `bootstrap=False` se hizo mediante **Cross-Validation interna** (K=5) dentro del GridSearchCV, que midió la generalización en datos no vistos en cada fold.
- El RMSE de CV (~49-53k) es consistente con el RMSE en Validation (~52k) y Test (~53k).
- La consistencia entre Validation y Test confirma que el modelo generaliza correctamente a datos nuevos.

### ¿Por qué Random Forest es el modelo ganador?

1. **Mejor RMSE en Validation y Cross-Validation:** ~49-53k vs ~60-70k de los modelos lineales.
2. **Mejor R²:** ~0.78-0.82 — explica ~80% de la variabilidad del precio.
3. **Cross-Validation confirma la elección:** La media de RMSE por CV es consistente con la evaluación directa.
4. **Feature Importance interpretable:** Permite entender qué variables impactan más el precio (ej: `median_income` domina).

### Resultado Final:
- **Modelo seleccionado:** `RandomForestRegressor`
- **Hiperparámetros óptimos:** Seleccionados via GridSearchCV con CV=5
- **RMSE en Test Set:** ~$53k USD (consistente con Validation → generaliza correctamente)
- **Conclusión:** El modelo final presenta **sobreajuste en sentido técnico** (RMSE Train=0 por `bootstrap=False`), pero su **capacidad de generalización está validada** por Cross-Validation, Validation Set y Test Set, cuyos RMSEs son consistentes entre sí. No tiene problemas de subajuste.